In [1]:
import pandas as pd


In [2]:
import numpy as np

# Read and merge dicom download files

Load the CSV file(s) from the DICOM download from LONI IDA. This is the image collection CSV that you created on LONI IDA. If you created multiple collections in LONI and downloaded them independently, they must be merged as shown in the example below. 

In [3]:
adni_all = pd.read_csv('/N/project/cfn-rady/andrea-dev/ADNI/LONI_download/meta_data/ADNI_go-3_MRI-fMRI-DTI_collection_2025-12-23.csv')


In [4]:
# If needed, merge all files into one big dataframe.
## If you have multiple csv files to merge, merge them two at a time as shown below.
# adni_all = pd.merge(adni_mri, adni_pet, how='outer')
# adni_all = pd.merge(adni_all, adni_dti, how='outer')


In [5]:
## Each row corresponds to a directory with DICOMs associated to a single Series with it's Image ID. 
#### The download sheet does NOT indicate how many dcm files are contained in each directory. 
print('This is the shape of the data-frame: ',adni_all.shape)
print(adni_all.columns)

This is the shape of the data-frame:  (129942, 12)
Index(['Image Data ID', 'Subject', 'Group', 'Sex', 'Age', 'Visit', 'Modality',
       'Description', 'Type', 'Acq Date', 'Format', 'Downloaded'],
      dtype='object')


In [6]:
# might need to fix merge error by changing visit code to object dtype.
adni_all['Visit'] = adni_all['Visit'].astype('object')

In [7]:
# Check the number of unique subjects. 
print('Unique Subject IDs: ',len(adni_all['Subject'].value_counts()))
# Check the number of unique Modalities. 
print('Modality types: ',adni_all['Modality'].value_counts())

Unique Subject IDs:  2623
Modality types:  Modality
MRI     113074
fMRI      8871
DTI       7997
Name: count, dtype: int64


In [8]:
# Check other fields
print('Other types: ',adni_all['Type'].value_counts())

Other types:  Type
Original    129942
Name: count, dtype: int64


^ Based on the CSV files from the LONI image collection downloads, there are **129942 DICOM directories** from **2623 unique subjects**.

In [9]:
# Chech the number of unique directory names for the DICOM directories.
unique_series = adni_all['Description'].value_counts()
print('Total Number of Unique Series Descriptions is ', len(unique_series))
print(unique_series[0:50])


Total Number of Unique Series Descriptions is  770
Description
3 Plane Localizer              12055
localizer                       7604
3-plane localizer               5823
MPRAGE                          5557
Field Mapping                   5510
Axial PD-T2 TSE                 3942
Axial PD/T2 FSE                 3822
Localizer                       3282
Axial T2-FLAIR                  3041
Perfusion_Weighted              2810
Axial DTI                       2780
Axial T2 Star                   2441
Sagittal 3D FLAIR               2310
Axial 3TE T2 STAR               2226
AXIAL_T2_STAR                   2075
MPRAGE GRAPPA2                  2061
MP-RAGE                         2037
MPRAGE Repeat                   1961
MoCoSeries                      1947
relCBF                          1935
B1-calibration Body             1897
B1-calibration Head             1866
MP-RAGE REPEAT                  1669
HighResHippo                    1487
B1-Calibration Body             1479
B1-Calibrati

In [10]:
adni_all[adni_all['Description'].str.contains('Sag IR-SPGR',case=False, na=False)];

# Building the resting state fMRI collection

In [11]:
# This is Saige's diectionary. Not exactly sure how she built it but I built my own basedon string matches
#  and I ended up with the a handful more series than if I had used her dictionary. 
# Build a dictionary of all possible fMRI DICOM directory names.
fmri_names_saige = ['Axial rsfMRI (Eyes Open)',
'Resting State fMRI',
'Axial MB rsfMRI (Eyes Open)',
'Axial fcMRI (EYES OPEN)',
'Axial rsfMRI (EYES OPEN)',
'Axial fcMRI (Eyes Open)',
'Extended Resting State fMRI',
'Axial rsfMRI (Eyes Open) -phase P to A',
'Axial fcMRI 0 angle (EYES OPEN)',
'Axial fcMRI',
'Axial MB rsfMRI (Eyes Open)   straight no angle',
'Axial RESTING fcMRI (EYES OPEN)',
'Axial MB rsfMRI AP',
'Axial rsfMRI (Eyes Open) 10 min :-PJ',
'Extended AXIAL rsfMRI EYES OPEN',
'Axial rsfMRI (Eyes Open) Phase Direction P>A',
'AXIAL RS fMRI (EYES OPEN)',
'Axial_rsFMRI_Eyes_Open',
'Extended Resting State fMRI CLEAR',
'Axial MB rsfMRI (Eyes Open) REPEAT',
'Axial fcMRI (EYES OPEN)_REPEAT',
'Axial rsfMRI P>>A (Eyes Open)',
'Axial MB rsfMRI',
'Axial MB rsfMRI (Eyes Open) (MSV21)',
'epi_2s_resting_state',
'Axial rsfMRI (Eyes Open) 10 min']

In [12]:
# I built my rsfMRI collection by finding all the series that include fMRI, fcMRI and epi in them. 
# I tried other substrings but these are the ones that I think cover the entire set
fMRIstr = adni_all[adni_all['Description'].str.contains('fMRI',case=False, na=False)]
fcMRIstr = adni_all[adni_all['Description'].str.contains('fcMRI',case=False, na=False)]
epistr = adni_all[adni_all['Description'].str.contains('epi',case=False, na=False)]
print('====== Matching fMRI string: ',len(fMRIstr))
print('======== Unique Series Descriptions: ',len(fMRIstr['Description'].unique()))
print(fMRIstr['Description'].value_counts())
print('====== Matching fcMRI string: ',len(fcMRIstr))
print('======== Unique Series Descriptions: ',len(fcMRIstr['Description'].unique()))
print(fcMRIstr['Description'].value_counts())
print('====== Matching epi string: ',len(epistr))
print(epistr['Description'].value_counts())
print('======== Unique Series Descriptions: ',len(epistr['Description'].unique()))
#print(fMRIstr)

====== Matching fMRI string:  2780
======== Unique Series Descriptions:  21
Description
Axial rsfMRI (Eyes Open)                           1051
Resting State fMRI                                  767
Axial MB rsfMRI (Eyes Open)                         424
Axial rsfMRI (EYES OPEN)                            295
Extended Resting State fMRI                         133
Axial rsfMRI (Eyes Open) -phase P to A               58
Axial MB rsfMRI (Eyes Open)   straight no angle      12
Axial MB rsfMRI AP                                    9
Axial rsfMRI (Eyes Open) 10 min :-PJ                  7
Axial rsfMRI (Eyes Open) Phase Direction P>A          5
Extended AXIAL rsfMRI EYES OPEN                       5
AXIAL RS fMRI (EYES OPEN)                             3
Axial rsfMRI (Eyes Open)(MSV21)                       2
Axial_rsFMRI_Eyes_Open                                2
Axial rsfMRI (Eyes Open) 10 min                       1
Extended Resting State fMRI CLEAR                     1
Axial MB rsfMRI 

In [13]:
# Concatenate the different matches
fmri_series_desc_names = np.concatenate([fMRIstr['Description'].unique(),fcMRIstr['Description'].unique(),epistr['Description'].unique()])
# remove potential repetitions
fmri_series_desc_names = np.unique(fmri_series_desc_names)
fmri_series_dirs = adni_all[adni_all['Description'].isin(fmri_series_desc_names)]
# Check the number of fMRI DICOM directories.
print('Number of DICOM directories found in collection: ',fmri_series_dirs.shape)
# Check the number of unique subjects who have an fMRI DICOM directory.
print('Number of Subjects with fMRI data: ',len(fmri_series_dirs['Subject'].value_counts()))

Number of DICOM directories found in collection:  (3304, 12)
Number of Subjects with fMRI data:  1308


In [14]:
# TO KEEP IN MIND: Don't rely on Modality tags to sort data - there is rsfMRI with non fMRI Modalities. 
fmri_series_dirs_test = adni_all[adni_all['Description'].isin(fmri_series_desc_names) & (adni_all['Modality'] != 'fMRI')]
print(fmri_series_dirs_test.shape)
# Check the number of unique subjects who have an fMRI DICOM directory.
print(len(fmri_series_dirs_test['Subject'].value_counts()))
print(fmri_series_dirs_test['Description'].value_counts())

(246, 12)
172
Description
Axial rsfMRI (Eyes Open)                  186
Axial MB rsfMRI (Eyes Open)                23
Axial rsfMRI (EYES OPEN)                   15
Axial rsfMRI (Eyes Open) -phase P to A     10
Axial fcMRI 0 angle (EYES OPEN)             4
Extended AXIAL rsfMRI EYES OPEN             2
Axial fcMRI (Eyes Open)                     2
Axial rsfMRI (Eyes Open) 10 min :-PJ        2
Axial rsfMRI P>>A (Eyes Open)               1
Axial rsfMRI (Eyes Open) 10 min             1
Name: count, dtype: int64


In [15]:
## Cross-checking with Saige's list of fMRI series descriptions. 
fmri_series_dirs_saige = adni_all[adni_all['Description'].isin(fmri_names_saige)]
# Check the number of fMRI DICOM directories.
print('shape of fmri_series_dirs_saige: ',fmri_series_dirs_saige.shape)
# Check the number of unique subjects who have an fMRI DICOM directory.
print('Number of unique subject IDs: ',len(fmri_series_dirs_saige['Subject'].value_counts()))

shape of fmri_series_dirs_saige:  (3301, 12)
Number of unique subject IDs:  1308


Pretty similar - Not sure where the discrepancy is coming from but it is just 3 folders that don't match

# Building the MPRAGE collection

In [21]:
test_mprage = adni_all[adni_all['Description'].str.contains('_nd',case=False, na=False)]
print('====== Matching mprage string: ',len(test_mprage))
print('======== Unique Series Descriptions: ',len(test_mprage['Description'].unique()))
test_mprage['Description'].unique()

====== Matching mprage string:  329
======== Unique Series Descriptions:  6


array(['Accelerated Sagittal MPRAGE_ND', 'B-LOC_S1_ND',
       'Field Mapping_ND', 'Localizer_S1_ND', 'MPRAGE_ND',
       'MPRAGE GRAPPA 2_ND'], dtype=object)

In [17]:
# I built the MPRAGE collection by finding all the series that include MPRAGE,MP-RAGE,SPGR and T1 
# I tried other substrings but these are the ones that I think cover the entire set
mprage = adni_all[adni_all['Description'].str.contains('MPRAGE',case=False, na=False)]
mp_rage = adni_all[adni_all['Description'].str.contains('MP-RAGE',case=False, na=False)]
SPGR = adni_all[adni_all['Description'].str.contains('SPGR',case=False, na=False)]
t1s = adni_all[adni_all['Description'].str.contains('T1',case=False, na=False)]
print('====== Matching mprage string: ',len(mprage))
print('======== Unique Series Descriptions: ',len(mprage['Description'].unique()))
#print(mprage['Description'].value_counts())
print('====== Matching mp-rage string: ',len(mp_rage))
print('======== Unique Series Descriptions: ',len(mp_rage['Description'].unique()))
#print(mp_rage['Description'].value_counts())
print('====== Matching SPGR string: ',len(SPGR))
print('======== Unique Series Descriptions: ',len(SPGR['Description'].unique()))
#print(SPGR['Description'].value_counts())
print('====== Matching T1 string: ',len(t1s))
print('======== Unique Series Descriptions: ',len(t1s['Description'].unique()))
#print(t1s['Description'].value_counts())
#print(fMRIstr)

====== Matching mprage string:  14736
======== Unique Series Descriptions:  115
====== Matching mp-rage string:  4208
======== Unique Series Descriptions:  25
====== Matching SPGR string:  3359
======== Unique Series Descriptions:  26
====== Matching T1 string:  12
======== Unique Series Descriptions:  4


In [18]:
# Concatenate the different matches
mprage_series_desc_names = np.concatenate([mprage['Description'].unique(),mp_rage['Description'].unique(),SPGR['Description'].unique(),t1s['Description'].unique()])
#print(len(mprage_series_desc_names))
mprage_series_desc_names = np.unique(mprage_series_desc_names)
print('Unique series descriptions for MPRAGE: ',len(mprage_series_desc_names))

mprage_series_dirs = adni_all[adni_all['Description'].isin(mprage_series_desc_names)]
# Check the number of fMRI DICOM directories.
print('Number of DICOM directories found in collection: ',mprage_series_dirs.shape)
# Check the number of unique subjects who have an fMRI DICOM directory.
print('Number of Subjects with T1 data: ',len(mprage_series_dirs['Subject'].value_counts()))

Unique series descriptions for MPRAGE:  168
Number of DICOM directories found in collection:  (22297, 12)
Number of Subjects with T1 data:  2620


In [20]:
mprage_series_dirs['Description'].value_counts()

Description
MPRAGE                       5557
MPRAGE GRAPPA2               2061
MP-RAGE                      2037
MPRAGE Repeat                1961
MP-RAGE REPEAT               1669
                             ... 
MP-RAGE (SERIES 2)              1
MP-RAGE REPEAT (SERIES 3)       1
SAG 3D MPRAGE, no angle         1
SAG MP-RAGE Repeat              1
MPRAGE_REPE                     1
Name: count, Length: 168, dtype: int64

# Building the DTI collection

In [18]:
test_dti = adni_all[adni_all['Description'].str.contains('diffusion',case=False, na=False)]
print('====== Matching dti string: ',len(test_dti))
print('======== Unique Series Descriptions: ',len(test_dti['Description'].unique()))
test_dti['Description'].unique()

====== Matching dti string:  7
======== Unique Series Descriptions:  4


array(['ADNI3 Advanced diffusion', 'Trace Diffusion ax_ADC',
       'Trace Diffusion ax', 'Apparent Diffusion Coefficient (mm2/s)'],
      dtype=object)

In [19]:
# I built the DTI collection by finding all the series that include MPRAGE,MP-RAGE 
# I tried other substrings but these are the ones that I think cover the entire set
dti = adni_all[adni_all['Description'].str.contains('DTI',case=False, na=False)]
print('====== Matching DTI string: ',len(dti))
print('======== Unique Series Descriptions: ',len(dti['Description'].unique()))
print(dti['Description'].value_counts())

====== Matching DTI string:  4587
======== Unique Series Descriptions:  57
Description
Axial DTI                                         2780
Axial MB DTI                                       411
Axial DTI_ADC                                      268
Axial DTI_FA                                       262
Axial DTI_TRACEW                                   252
Enhanced Axial DTI                                 114
Axial DTI - phase P to A (180 degrees)              76
Reg - Axial DTI                                     50
Axial DTI straight                                  38
AX DTI                                              33
Axial MB DTI_TRACEW                                 30
Axial DTI SEE NOTE)                                 30
Axial DTI :-PJ                                      23
Axial MB DTI phase_noFatSatA                        18
Axial MB DTI_ADC                                    18
Axial MB DTI_FA                                     18
Axial MB DTI_TENSOR_B0           

In [20]:
# Concatenate the different matches
# dti_series_desc_names = np.concatenate([mprage['Description'].unique(),])
#print(len(dti_series_desc_names))
dti_series_desc_names = np.unique(dti['Description'].unique())
print('Unique series descriptions for DTI: ',len(dti_series_desc_names))

dti_series_dirs = adni_all[adni_all['Description'].isin(dti_series_desc_names)]
# Check the number of DTI DICOM directories.
print('Number of DICOM directories found in collection: ',dti_series_dirs.shape)
# Check the number of unique subjects who have a DTI DICOM directory.
print('Number of Subjects with fMRI data: ',len(dti_series_dirs['Subject'].value_counts()))

Unique series descriptions for DTI:  57
Number of DICOM directories found in collection:  (4587, 12)
Number of Subjects with fMRI data:  1333


In [21]:
# save the merge DICOM directory file.
# adni_all.to_csv('/N/project/cfn-rady/andrea-dev/ADNI/LONI_data/scripts/ADNI_go-3_merged.csv', index=False)

## Read the csv listing all the files extracted from the zip downloads from LONI

Before loading the file that lists all the unzipped DICOM data, I had to do some editing of the Series Descriptions. 
Specifically, the CSV provided by LONI has removed underscores and parenthesis from the description names. 

First I made a copy of the CSV file so that I could preserve the original file names as they appear in my unzipped DICOM data. I added `_edited.csv` to the copy to distinguish the files. 

Then I opened up the edited CSV file with VIM and did the following substitutions in this order:
 - `:%s/_EYES_OPEN_/(EYES OPEN)/g`  => adds parenthesys around the `open eyes` strings
 - `:%s/_Eyes_Open_/(Eyes Open)/g`
 - `:%s/_/ /g`                      => substitutes all underscores for spaces
 - `:%s/  -PJ/ :-PJ/g`               => adds colon to the PJ strings
 - `:%s/Direction P A/Direction P>A/g`  => adds the > for the P=>A direction 
 - `:%s/w acceleration/w\/acceleration/g`
 - `:%s/MPRAGE S2 DIS3D/MPRAGE_S2_DIS3D/g`
 - `:%s/MPRAGE ASO repeat/MPRAGE_ASO_repeat/n`
 
 
 
 This worked for resting state fMRI. I haven't yet checked other series.  

In [61]:
# Load the file created from just listing the unzipped DICOM directories.
# I did this in an attempt to verify the download. 
# This list should match the all_data list if the download was successful.
adni_unzip_path="/N/project/cfn-rady/andrea-dev/ADNI/LONI_data/image_data/ADNI/unzipped_dicom_dirs_inventory_edited.csv"
adni_unzip=pd.read_csv(adni_unzip_path)
print(adni_unzip.columns)
print('Number of Series Directories unzipped: ',adni_unzip.shape)
# Check the number of unique subjects. 
print('Unique Subject IDs: ',len(adni_unzip['Subject'].value_counts()))

Index(['Subject', 'Description', 'Acq Date', 'ImageID'], dtype='object')
Number of Series Directories unzipped:  (129942, 4)
Unique Subject IDs:  2623


Number of Directories and Subject IDs matches what we expected from the collection download metadata

Test if the edits I made to series descriptin names align the with the descriptions from the LONI downloaded metadata

In [62]:
## fMRI
fmri_series_dirs_unzip = adni_unzip[adni_unzip['Description'].isin(fmri_series_desc_names)]
# Check the number of fMRI DICOM directories.
print('Number of DICOM directories unzipped: ',fmri_series_dirs_unzip.shape)
print('Number of unzipped Subjects with fMRI data: ',len(fmri_series_dirs_unzip['Subject'].value_counts()))

print('Number of directories listed in metadata: ',fmri_series_dirs.shape)
print('Number of Subjects listed in metadata: ',len(fmri_series_dirs['Subject'].value_counts()))

#print([fmri_series_dirs_unzip['Description'].value_counts(),fmri_series_dirs['Description'].value_counts()])

Number of DICOM directories unzipped:  (3295, 4)
Number of unzipped Subjects with fMRI data:  1306
Number of directories listed in metadata:  (3304, 12)
Number of Subjects listed in metadata:  1308


In [63]:
## Test if the edits I made to series descriptin names align the with the descriptions from the LONI downloaded metadata
fMRIstr_zip = adni_unzip[adni_unzip['Description'].str.contains('fMRI',case=False, na=False)]
fcMRIstr_zip = adni_unzip[adni_unzip['Description'].str.contains('fcMRI',case=False, na=False)]
epistr_zip = adni_unzip[adni_unzip['Description'].str.contains('epi',case=False, na=False)]
print('====== Matching fMRI string: ',len(fMRIstr_zip))
print('======== Unique Series Descriptions: ',len(fMRIstr_zip['Description'].unique()))
#print(fMRIstr_zip['Description'].value_counts())
print('====== Matching fcMRI string: ',len(fcMRIstr_zip))
print('======== Unique Series Descriptions: ',len(fcMRIstr_zip['Description'].unique()))
#print(fcMRIstr_zip['Description'].value_counts())
print('====== Matching epi string: ',len(epistr_zip))
print(epistr_zip['Description'].value_counts())
#print('======== Unique Series Descriptions: ',len(epistr_zip['Description'].unique()))
#print(fMRIstr)

====== Matching fMRI string:  2780
======== Unique Series Descriptions:  21
====== Matching fcMRI string:  523
======== Unique Series Descriptions:  6
====== Matching epi string:  1
Description
epi 2s resting state    1
Name: count, dtype: int64


In [66]:
## MPRAGE
mprage_series_dirs_unzip = adni_unzip[adni_unzip['Description'].isin(mprage_series_desc_names)]
# Check the number of fMRI DICOM directories.
print('Number of DICOM directories unzipped: ',mprage_series_dirs_unzip.shape)
print('Number of unzipped Subjects with MPRAGE data: ',len(mprage_series_dirs_unzip['Subject'].value_counts()))

print('Number of directories listed in metadata: ',mprage_series_dirs.shape)
print('Number of Subjects listed in metadata: ',len(mprage_series_dirs['Subject'].value_counts()))

#print([mprage_series_dirs_unzip['Description'].value_counts()[0:50],mprage_series_dirs['Description'].value_counts()[0:50]])

Number of DICOM directories unzipped:  (21823, 4)
Number of unzipped Subjects with MPRAGE data:  2620
Number of directories listed in metadata:  (22297, 12)
Number of Subjects listed in metadata:  2620


There are some differences due to how LONI has edited the series descriptions in the downloaded collection metadata. 

In [127]:
fmri_series_desc_names_matt = np.concatenate([fMRIstr_matt['Description'].unique(),fcMRIstr_matt['Description'].unique(),epistr_matt['Description'].unique()])
print(len(fmri_series_desc_names_matt))

NameError: name 'fMRIstr_matt' is not defined

In [95]:
fmri_all_matt1 = adni_matt[adni_matt['Description'].isin(fmri_series_desc_names_matt)]
# Check the number of fMRI DICOM directories.
print(fmri_all_matt1.shape)
# Check the number of unique subjects who have an fMRI DICOM directory.
print('Number of Subjects with fMRI data: ',len(fmri_all_matt1['subID'].value_counts()))

(2444, 4)
Number of Subjects with fMRI data:  793


In [96]:
fmri_all_matt2 = adni_matt[adni_matt['Description'].isin(fmri_series_desc_names)]
# Check the number of fMRI DICOM directories.
print(fmri_all_matt2.shape)
# Check the number of unique subjects who have an fMRI DICOM directory.
print('Number of Subjects with fMRI data: ',len(fmri_all_matt2['Subject'].value_counts()))

(2425, 4)
Number of Subjects with fMRI data:  789


In [102]:
#adni_matt[adni_matt['series_desc'].isin(fmri_series_desc_names)]
print(len(adni_matt['Description'].value_counts()))
unique_series_matt=adni_matt['Description'].value_counts()
#print(unique_series_matt[0:20])

1598


^ Based on the file I created by listing the unzipped DICOM directories, there are **29551 DICOM directories** from **2018 unique subjects**.

The number of unique subjects (2018) matches, but there seems to be a lot more DICOM directories in the LONI IDA csv files than in the unzipped DICOM directories. 

In [110]:
print(fmri_all_matt1['Description'].value_counts())
print(fmri_all_matt2['Description'].value_counts())


SeriesDescription
Resting State fMRI                                 721
Axial rsfMRI (Eyes Open)                           714
Axial MB rsfMRI (Eyes Open)                        273
Axial fcMRI (EYES OPEN)                            213
Axial rsfMRI (EYES OPEN)                           197
Extended Resting State fMRI                        125
Axial fcMRI (Eyes Open)                            115
Axial rsfMRI (Eyes Open) -phase P to A              18
Axial RESTING fcMRI (EYES OPEN)                     11
Axial MB rsfMRI (Eyes Open)   straight no angle     10
Axial fcMRI                                          9
Axial fcMRI 0 angle (EYES OPEN)                      7
Axial rsfMRI (Eyes Open) 10 min  -PJ                 6
Axial rsfMRI (Eyes Open) Phase Direction P A         5
Extended AXIAL rsfMRI EYES OPEN                      5
AXIAL RS fMRI (EYES OPEN)                            3
Axial rsfMRI (Eyes Open)  MSV21                      2
Axial MB rsfMRI (Eyes Open) REPEAT             

In [ ]:
# This cell was used to list out the DICOM directory names.
# Then I had to manually go through and chose the names that are for fMRI scans, which I input to the dictionary in the cell below.
#pd.set_option('display.max_rows', None)

In [ ]:
# Build a dictionary of all the fMRI DICOM directory names
fmri_dcm = ['Axial_rsfMRI__Eyes_Open_','Resting_State_fMRI','Axial_MB_rsfMRI__Eyes_Open_', 'Axial_fcMRI__EYES_OPEN_','Axial_rsfMRI__EYES_OPEN_',
'Extended_Resting_State_fMRI','Axial_fcMRI__Eyes_Open_', 'Axial_rsfMRI__Eyes_Open__-phase_P_to_A', 'Axial_RESTING_fcMRI__EYES_OPEN_',
'Axial_rsfMRI__Eyes_Open__10_min__-PJ','Axial_MB_rsfMRI__Eyes_Open____straight_no_angle','Axial_rsfMRI__Eyes_Open__Phase_Direction_P_A',
'Extended_AXIAL_rsfMRI_EYES_OPEN','AXIAL_RS_fMRI__EYES_OPEN_', 'Axial_rsFMRI_Eyes_Open','Axial_MB_rsfMRI__Eyes_Open__REPEAT',
'Axial_rsfMRI__Eyes_Open__10_min','epi_2s_resting_state','Axial_fcMRI__EYES_OPEN__REPEAT','Axial_MB_rsfMRI__Eyes_Open___MSV21_',
'Axial_MB_rsfMRI','Axial_-_Advanced_fMRI_64_Channel','Axial_rsfMRI_P__A__Eyes_Open_']

In [ ]:
# Build a new dataframe that only contains the fMRI DICOM directories.
fmri_dcm_df = dcm_df[dcm_df['scan_name'].isin(fmri_dcm)]

In [ ]:
# Check the number of fMRI DICOM directories.
fmri_dcm_df.shape

(1901, 7)

In [ ]:
# Check the number of unique subjects with fMRI DICOM directory.
len(fmri_dcm_df['Subject_ID'].value_counts())

1284

^ Based on the file I created by listing the unzipped DICOM directories, there are **1901 fMRI DICOM directories** from **1284 unique subjects**.

In [ ]:
# add line here to make adni_subs.txt file to pass to Clinica script. 